In [6]:
pip install transformer_lens torch

In [7]:
"""
milestone2_direct_path_primitive.py
===================================
Isolated Direct Path Patching Primitive for GPT-2 Small.
Manipulates ONLY the Value Vector (hook_v) of Target Head B corresponding
to the contribution of Source Head A (z_A @ W_O_A @ W_V_B).
Guarantees zero leakage to sibling heads or MLPs in the target layer.
"""

import torch
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Audit Schema for Isolated Edge
# =============================================================================

@dataclass
class CausalEdgeAudit:
    source: Tuple[int, int]
    target: Tuple[int, int]
    site: str
    baseline_diff: float
    patched_diff: float
    direct_effect: float
    null_effect: float
    is_valid: bool


# =============================================================================
# 2. Isolated Direct Path Primitive Functions
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    """محاسبه سهم برداری اختصاصی Source Head در Residual Stream: z @ W_O"""
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]  # [d_head]
    w_o = model.W_O[layer, head]                               # [d_head, d_model]
    return z @ w_o                                             # [d_model]


def patch_isolated_target_v_hook(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> float:
    """
    مداخله ایزوله‌شده: تزریق تغییرات سهم Source A فقط روی بردار Value هد Target B.
    delta_v_B = (contrib_clean_A - contrib_corrupted_A) @ W_V_B
    """
    l_src, h_src = source
    l_tgt, h_tgt = target

    assert l_src < l_tgt, f"Temporal Assertion Failed: Source L{l_src} >= Target L{l_tgt}"

    # ۱. استخراج سهم Source A در دو حالت
    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    # ۲. تصویر کردن سهم Source A روی فضای Value اختصاصی Target Head B
    w_v_target = model.W_V[l_tgt, h_tgt]       # [d_model, d_head]
    delta_v_target = delta_contrib @ w_v_target # [d_head]

    # ۳. تعریف Hook ایزوله روی hook_v که فقط خروجی V هد Target را تغییر می‌دهد
    def patch_v_hook(act, hook):
        # act shape: [batch, seq_len, n_heads, d_head]
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_v_target
        return act

    hook_site = f"blocks.{l_tgt}.attn.hook_v"
    hooks = [(hook_site, patch_v_hook)]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    return float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())


# =============================================================================
# 3. Execution & Validation Protocol
# =============================================================================

def run_primitive_audit():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = HookedTransformer.from_pretrained("gpt2", device=device)
    model.eval()

    task = {
        "clean": "Then, John and Mary went to the store. John gave a drink to",
        "corrupted": "Then, John and Mary went to the store. Mary gave a drink to",
        "correct": " Mary",
        "incorrect": " John"
    }

    clean_toks = model.to_tokens(task["clean"])
    corrupted_toks = model.to_tokens(task["corrupted"])

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        _, clean_cache = model.run_with_cache(clean_toks)
        base_logits, corrupted_cache = model.run_with_cache(corrupted_toks)

    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

    # Candidate Pair: L10H7 (Negative Name Mover) -> L11H10
    pos_src, pos_tgt = (10, 7), (11, 10)
    null_src = (0, 2)
    temporal_src = (11, 2)

    print("==================================================================")
    print("🔬 ISOLATED DIRECT PATH PATCHING AUDIT (hook_v Geometry)")
    print("==================================================================")
    print(f"Base Corrupted Logit Difference: {base_diff:.4f}\n")

    # Test 1: Positive Candidate
    patched_pos = patch_isolated_target_v_hook(model, pos_src, pos_tgt, task, clean_cache, corrupted_cache, corrupted_toks)
    pos_effect = patched_pos - base_diff
    print(f"1. Candidate Pair (L{pos_src[0]}H{pos_src[1]} -> L{pos_tgt[0]}H{pos_tgt[1]} via V_target):")
    print(f"   Patched Logit Diff: {patched_pos:.4f} | Isolated Direct Effect: {pos_effect:+6.4f}")

    # Test 2: Random Control
    patched_null = patch_isolated_target_v_hook(model, null_src, pos_tgt, task, clean_cache, corrupted_cache, corrupted_toks)
    null_effect = patched_null - base_diff
    print(f"\n2. Null Control (L{null_src[0]}H{null_src[1]} -> L{pos_tgt[0]}H{pos_tgt[1]}):")
    print(f"   Patched Logit Diff: {patched_null:.4f} | Null Direct Effect: {null_effect:+6.4f}")

    # Test 3: Temporal Control
    print(f"\n3. Temporal Control (L{temporal_src[0]} -> L10):")
    try:
        patch_isolated_target_v_hook(model, temporal_src, pos_src, task, clean_cache, corrupted_cache, corrupted_toks)
    except AssertionError as e:
        print(f"   ✓ Caught Temporal Violation: {e}")

    is_valid = abs(pos_effect) > 0.05 and abs(null_effect) < 0.02
    print("\n------------------------------------------------------------------")
    print(f"🎯 PRIMITIVE AUDIT VERDICT: {'PASSED ✅' if is_valid else 'FAILED ❌'}")
    print("------------------------------------------------------------------")

if __name__ == "__main__":
    run_primitive_audit()

/tmp/ipykernel_803/116536067.py:95: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
🔬 ISOLATED DIRECT PATH PATCHING AUDIT (hook_v Geometry)
Base Corrupted Logit Difference: -3.1417

1. Candidate Pair (L10H7 -> L11H10 via V_target):
   Patched Logit Diff: -3.1973 | Isolated Direct Effect: -0.0556

2. Null Control (L0H2 -> L11H10):
   Patched Logit Diff: -3.1417 | Null Direct Effect: -0.0000

3. Temporal Control (L11 -> L10):
   ✓ Caught Temporal Violation: Temporal Assertion Failed: Source L11 >= Target L10

------------------------------------------------------------------
🎯 PRIMITIVE AUDIT VERDICT: PASSED ✅
------------------------------------------------------------------


In [8]:
"""
milestone2_circuit_discovery_engine.py
======================================
Milestone 2: Causal Circuit Discovery Engine on GPT-2 Small.
Fixed: ActivationCache assignment TypeError resolved.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Causal Edge & Circuit Graph Schema
# =============================================================================

@dataclass
class CausalEdge:
    source: Tuple[int, int]         # (layer, head)
    target: Tuple[int, int]         # (layer, head)
    intervention_site: str          # e.g., "blocks.11.attn.hook_v"

    direct_effect: float            # اثر مداخله ایزوله روی logit diff
    necessity_score: float          # میزان افت عملکرد پس از حذف این مسیر

    is_causal: bool
    is_direct: bool
    is_necessary: bool

    null_control_effect: float
    regime_id: str


@dataclass
class SelfCircuitGraph:
    regime_id: str
    nodes: Set[Tuple[int, int]] = field(default_factory=set)
    edges: List[CausalEdge] = field(default_factory=list)
    output_target: str = "Logit Difference (Correct - Incorrect)"

    def add_edge(self, edge: CausalEdge):
        self.nodes.add(edge.source)
        self.nodes.add(edge.target)
        self.edges.append(edge)

    def print_circuit_summary(self):
        print("\n==================================================")
        print(f"🕸️ DISCOVERED SELFCIRCUITGRAPH FOR REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Total Discovered Active Nodes: {len(self.nodes)}")
        print(f"Total Verified Causal Edges: {len(self.edges)}\n")

        for edge in self.edges:
            status_flags = []
            if edge.is_direct: status_flags.append("DIRECT")
            if edge.is_necessary: status_flags.append("NECESSARY")
            flags_str = ", ".join(status_flags) if status_flags else "WEAK"

            print(f"  ★ Edge L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | "
                  f"Direct Delta: {edge.direct_effect:+6.4f} | "
                  f"Necessity: {edge.necessity_score:+6.4f} | "
                  f"[{flags_str}]")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations: Isolated Patching & Edge Ablation
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_isolated_direct_edge(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> float:
    l_src, h_src = source
    l_tgt, h_tgt = target

    assert l_src < l_tgt, f"Temporal Violation: L{l_src} >= L{l_tgt}"

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    w_v_target = model.W_V[l_tgt, h_tgt]
    delta_v_target = delta_contrib @ w_v_target

    def patch_v_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_v_target
        return act

    hook_site = f"blocks.{l_tgt}.attn.hook_v"
    hooks = [(hook_site, patch_v_hook)]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    return float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())


def evaluate_edge_necessity_ablation(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    clean_logits: torch.Tensor,
    clean_toks: torch.Tensor,
    pos: int = -1
) -> float:
    l_src, h_src = source
    l_tgt, h_tgt = target

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    w_v_target = model.W_V[l_tgt, h_tgt]
    v_to_zero = contrib_clean @ w_v_target

    def ablate_v_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] - v_to_zero
        return act

    hook_site = f"blocks.{l_tgt}.attn.hook_v"
    hooks = [(hook_site, ablate_v_hook)]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        ablated_logits = model.run_with_hooks(clean_toks, fwd_hooks=hooks)

    clean_diff = float((clean_logits[0, pos, correct_id] - clean_logits[0, pos, incorrect_id]).item())
    ablated_diff = float((ablated_logits[0, pos, correct_id] - ablated_logits[0, pos, incorrect_id]).item())

    return clean_diff - ablated_diff


# =============================================================================
# 3. Milestone 2 Circuit Discovery Engine
# =============================================================================

class CircuitDiscoveryEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()

    def discover_circuit(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        direct_threshold: float = 0.03,
        necessity_threshold: float = 0.05
    ) -> SelfCircuitGraph:

        circuit_graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [MILESTONE 2] Running Circuit Discovery for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, clean_logits, base_corrupted_logits))

        for i in range(len(sorted_nodes)):
            for j in range(i + 1, len(sorted_nodes)):
                src, tgt = sorted_nodes[i], sorted_nodes[j]

                if src[0] >= tgt[0]:
                    continue

                direct_effects = []
                necessity_scores = []
                null_effects = []

                null_src = (0, 2)

                for task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, clean_logits, base_logits in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    patched_diff = evaluate_isolated_direct_edge(self.model, src, tgt, task, clean_cache, corrupted_cache, corrupted_toks)
                    direct_effects.append(patched_diff - base_diff)

                    nec_score = evaluate_edge_necessity_ablation(self.model, src, tgt, task, clean_cache, clean_logits, clean_toks)
                    necessity_scores.append(nec_score)

                    null_patched = evaluate_isolated_direct_edge(self.model, null_src, tgt, task, clean_cache, corrupted_cache, corrupted_toks)
                    null_effects.append(null_patched - base_diff)

                avg_direct = float(np.mean(direct_effects))
                avg_necessity = float(np.mean(necessity_scores))
                avg_null = float(np.mean(null_effects))

                is_causal = abs(avg_direct) >= direct_threshold
                is_direct = is_causal and (abs(avg_direct) > 2 * abs(avg_null))
                is_necessary = abs(avg_necessity) >= necessity_threshold

                if is_causal or is_necessary:
                    edge = CausalEdge(
                        source=src,
                        target=tgt,
                        intervention_site=f"blocks.{tgt[0]}.attn.hook_v",
                        direct_effect=avg_direct,
                        necessity_score=avg_necessity,
                        is_causal=is_causal,
                        is_direct=is_direct,
                        is_necessary=is_necessary,
                        null_control_effect=avg_null,
                        regime_id=regime_id
                    )
                    circuit_graph.add_edge(edge)

        return circuit_graph


# =============================================================================
# 4. Execution Pipeline
# =============================================================================

if __name__ == "__main__":
    engine = CircuitDiscoveryEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    circuit_graph = engine.discover_circuit(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        direct_threshold=0.03,
        necessity_threshold=0.05
    )

    circuit_graph.print_circuit_summary()

/tmp/ipykernel_803/3488364548.py:158: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 2] Running Circuit Discovery for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ DISCOVERED SELFCIRCUITGRAPH FOR REGIME 'R1_IOI'
Total Discovered Active Nodes: 3
Total Verified Causal Edges: 2

  ★ Edge L9H9  ──► L11H10 | Direct Delta: +0.0469 | Necessity: +0.0210 | [DIRECT]
  ★ Edge L10H7  ──► L11H10 | Direct Delta: -0.0424 | Necessity: -0.0151 | [DIRECT]



In [ ]:
"""
milestone2_level1_value_influence_v2.py
=======================================
Milestone 2 - Level 1: Target-Value Influence Test.
Includes:
1. Candidate Exclusion from Null Distribution (prevents contamination).
2. Discovery Threshold metric (Null-standardized Score).
3. Immutable Cache fix for TransformerLens.
"""

import torch
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Optional
from transformer_lens import HookedTransformer

# =============================================================================
# 1. Multi-Level Evidence Schema
# =============================================================================

@dataclass
class EvidenceChain:
    raw_influence_effect: float
    null_mean: float
    null_std: float
    null_standardized_score: float

    # Levels 2 & 3 reserved
    direct_path_verified: bool = False
    necessity_verified: bool = False


@dataclass
class CausalEdge:
    source: Tuple[int, int]
    target: Tuple[int, int]
    regime_id: str
    evidence: EvidenceChain

    @property
    def classification(self) -> str:
        if self.evidence.necessity_verified and self.evidence.direct_path_verified:
            return "VERIFIED_NECESSARY_DIRECT_EDGE"
        elif self.evidence.direct_path_verified:
            return "VERIFIED_DIRECT_EDGE"
        elif abs(self.evidence.null_standardized_score) >= 3.0:
            return "DIRECTIONAL_VALUE_INFLUENCE"
        return "CANDIDATE_ONLY"


@dataclass
class SelfCircuitGraph:
    regime_id: str
    edges: List[CausalEdge] = field(default_factory=list)

    def print_circuit_summary(self, candidate_nodes_count: int):
        print("\n==================================================")
        print(f"🕸️ SELFCIRCUITGRAPH (EVIDENCE LEVEL 1) FOR REGIME '{self.regime_id}'")
        print("==================================================")
        print(f"Candidate Nodes Evaluated: {candidate_nodes_count}")

        directional_count = sum(1 for e in self.edges if e.classification == "DIRECTIONAL_VALUE_INFLUENCE")
        direct_count = sum(1 for e in self.edges if e.classification == "VERIFIED_DIRECT_EDGE")
        necessary_count = sum(1 for e in self.edges if e.classification == "VERIFIED_NECESSARY_DIRECT_EDGE")

        print(f"Directional Value Influences Detected (|Z| >= 3.0): {directional_count}")
        print(f"Direct Edges Verified: {direct_count}")
        print(f"Necessary Edges Verified: {necessary_count}\n")

        for edge in self.edges:
            ev = edge.evidence
            print(f"  ★ Pair L{edge.source[0]}H{edge.source[1]:<2} ──► L{edge.target[0]}H{edge.target[1]:<2} | "
                  f"Raw Delta: {ev.raw_influence_effect:+6.4f} | "
                  f"Null Mean: {ev.null_mean:+6.4f} (std: {ev.null_std:.4f}) | "
                  f"Null-Std Score: {ev.null_standardized_score:+6.2f} | "
                  f"[{edge.classification}]")
        print("==================================================\n")


# =============================================================================
# 2. Primitive Operations
# =============================================================================

def compute_head_contribution(model: HookedTransformer, cache: Dict, layer: int, head: int, pos: int = -1) -> torch.Tensor:
    z = cache[f"blocks.{layer}.attn.hook_z"][0, pos, head, :]
    w_o = model.W_O[layer, head]
    return z @ w_o


def evaluate_target_value_intervention(
    model: HookedTransformer,
    source: Tuple[int, int],
    target: Tuple[int, int],
    task: Dict,
    clean_cache: Dict,
    corrupted_cache: Dict,
    corrupted_toks: torch.Tensor,
    pos: int = -1
) -> float:
    l_src, h_src = source
    l_tgt, h_tgt = target

    assert l_src < l_tgt, f"Temporal Violation: L{l_src} >= L{l_tgt}"

    contrib_clean = compute_head_contribution(model, clean_cache, l_src, h_src, pos)
    contrib_corrupted = compute_head_contribution(model, corrupted_cache, l_src, h_src, pos)
    delta_contrib = contrib_clean - contrib_corrupted

    w_v_target = model.W_V[l_tgt, h_tgt]
    delta_v_target = delta_contrib @ w_v_target

    def patch_v_hook(act, hook):
        act[0, pos, h_tgt, :] = act[0, pos, h_tgt, :] + delta_v_target
        return act

    hook_site = f"blocks.{l_tgt}.attn.hook_v"
    hooks = [(hook_site, patch_v_hook)]

    correct_id = int(model.to_single_token(task["correct"]))
    incorrect_id = int(model.to_single_token(task["incorrect"]))

    with torch.no_grad():
        patched_logits = model.run_with_hooks(corrupted_toks, fwd_hooks=hooks)

    return float((patched_logits[0, pos, correct_id] - patched_logits[0, pos, incorrect_id]).item())


# =============================================================================
# 3. Level 1 Evaluation Engine with Clean Null Sampling
# =============================================================================

class Level1InfluenceEngine:
    def __init__(self, model_name: str = "gpt2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = HookedTransformer.from_pretrained(model_name, device=self.device)
        self.model.eval()

    def generate_clean_null_distribution(
        self,
        target: Tuple[int, int],
        task_data: List,
        exclude_sources: Set[Tuple[int, int]],
        k_samples: int = 8
    ) -> Tuple[float, float]:
        """تولید توزیع نویز تمیز با حذف تمام هدهای کاندید مربوط به آن Target"""
        l_tgt, _ = target

        # انتخاب هدهای تصادفی از لایه‌های قبل، با استثنا کردن کاندیدها
        possible_null_sources = [
            (l, h) for l in range(l_tgt) for h in range(12)
            if (l, h) not in exclude_sources
        ]

        if len(possible_null_sources) < k_samples:
            null_sources = possible_null_sources
        else:
            np.random.seed(42)
            indices = np.random.choice(len(possible_null_sources), k_samples, replace=False)
            null_sources = [possible_null_sources[idx] for idx in indices]

        null_deltas = []
        for src in null_sources:
            task_effects = []
            for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits in task_data:
                correct_id = int(self.model.to_single_token(task["correct"]))
                incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                patched_diff = evaluate_target_value_intervention(
                    self.model, src, target, task, clean_cache, corrupted_cache, corrupted_toks
                )
                task_effects.append(patched_diff - base_diff)
            null_deltas.append(float(np.mean(task_effects)))

        return float(np.mean(null_deltas)), float(np.std(null_deltas) + 1e-8)

    def evaluate_level1_influences(
        self,
        regime_id: str,
        candidate_nodes: List[Tuple[int, int]],
        tasks: List[Dict],
        k_null_samples: int = 8
    ) -> SelfCircuitGraph:

        circuit_graph = SelfCircuitGraph(regime_id=regime_id)
        sorted_nodes = sorted(candidate_nodes, key=lambda x: x[0])

        print(f"\n⚡️ [MILESTONE 2 - LEVEL 1] Running Clean Target-Value Influence Scan for Regime '{regime_id}'")
        print(f"   Candidate Nodes: {sorted_nodes}")

        task_data = []
        for task in tasks:
            clean_toks = self.model.to_tokens(task["clean"])
            corrupted_toks = self.model.to_tokens(task["corrupted"])

            with torch.no_grad():
                clean_logits, clean_cache = self.model.run_with_cache(clean_toks)
                base_corrupted_logits, corrupted_cache = self.model.run_with_cache(corrupted_toks)

            task_data.append((task, clean_toks, corrupted_toks, clean_cache, corrupted_cache, base_corrupted_logits))

        # ساخت مجموعه کاندیدهای پیش از هر Target برای Exclusion
        for j in range(len(sorted_nodes)):
            tgt = sorted_nodes[j]
            # تمام کاندیدهایی که لایه آن‌ها قبل از Target است
            target_candidates = {sorted_nodes[i] for i in range(j) if sorted_nodes[i][0] < tgt[0]}

            for i in range(j):
                src = sorted_nodes[i]
                if src[0] >= tgt[0]:
                    continue

                # 1. محاسبه اثر Raw Candidate Pair
                raw_effects = []
                for task, _, corrupted_toks, clean_cache, corrupted_cache, base_logits in task_data:
                    correct_id = int(self.model.to_single_token(task["correct"]))
                    incorrect_id = int(self.model.to_single_token(task["incorrect"]))
                    base_diff = float((base_logits[0, -1, correct_id] - base_logits[0, -1, incorrect_id]).item())

                    patched_diff = evaluate_target_value_intervention(
                        self.model, src, tgt, task, clean_cache, corrupted_cache, corrupted_toks
                    )
                    raw_effects.append(patched_diff - base_diff)

                avg_raw = float(np.mean(raw_effects))

                # 2. محاسبه توزیع Null تمیز با استثنا کردن تمام Candidateهای مربوط به این Target
                null_mean, null_std = self.generate_clean_null_distribution(
                    target=tgt,
                    task_data=task_data,
                    exclude_sources=target_candidates,
                    k_samples=k_null_samples
                )

                # 3. محاسبه Null-Standardized Score
                null_std_score = (avg_raw - null_mean) / null_std

                evidence = EvidenceChain(
                    raw_influence_effect=avg_raw,
                    null_mean=null_mean,
                    null_std=null_std,
                    null_standardized_score=null_std_score
                )

                edge = CausalEdge(source=src, target=tgt, regime_id=regime_id, evidence=evidence)
                circuit_graph.edges.append(edge)

        return circuit_graph


# =============================================================================
# 4. Execution Pipeline
# =============================================================================

if __name__ == "__main__":
    engine = Level1InfluenceEngine("gpt2")

    IOI_TASKS = [
        {"clean": "Then, John and Mary went to the store. John gave a drink to", "corrupted": "Then, John and Mary went to the store. Mary gave a drink to", "correct": " Mary", "incorrect": " John"},
        {"clean": "Then, Alice and Bob went to the park. Alice gave a toy to", "corrupted": "Then, Alice and Bob went to the park. Bob gave a toy to", "correct": " Bob", "incorrect": " Alice"}
    ]

    top_ioi_candidate_nodes = [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

    circuit_graph = engine.evaluate_level1_influences(
        regime_id="R1_IOI",
        candidate_nodes=top_ioi_candidate_nodes,
        tasks=IOI_TASKS,
        k_null_samples=8
    )

    circuit_graph.print_circuit_summary(candidate_nodes_count=len(top_ioi_candidate_nodes))

/tmp/ipykernel_803/3394619567.py:135: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  self.model = HookedTransformer.from_pretrained(model_name, device=self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

⚡️ [MILESTONE 2 - LEVEL 1] Running Clean Target-Value Influence Scan for Regime 'R1_IOI'
   Candidate Nodes: [(7, 9), (8, 6), (8, 10), (9, 9), (10, 7), (11, 10)]

🕸️ SELFCIRCUITGRAPH (EVIDENCE LEVEL 1) FOR REGIME 'R1_IOI'
Candidate Nodes Evaluated: 6
Directional Value Influences Detected (|Z| >= 3.0): 7
Direct Edges Verified: 0
Necessary Edges Verified: 0

  ★ Pair L7H9  ──► L8H6  | Raw Delta: +0.0000 | Null Mean: +0.0000 (std: 0.0000) | Null-Std Score:  +0.45 | [CANDIDATE_ONLY]
  ★ Pair L7H9  ──► L8H10 | Raw Delta: -0.0044 | Null Mean: +0.0020 (std: 0.0048) | Null-Std Score:  -1.33 | [CANDIDATE_ONLY]
  ★ Pair L7H9  ──► L9H9  | Raw Delta: -0.0000 | Null Mean: +0.0000 (std: 0.0000) | Null-Std Score:  -6.36 | [DIRECTIONAL_VALUE_INFLUENCE]
  ★ Pair L8H6  ──► L9H9  | Raw Delta: -0.0000 | Null Mean: +0.0000 (std: 0.0000) | Null-Std Score:  -1.01 | [CANDIDATE_ONLY]
  ★ Pair L8H10 ──► L9H9  | Raw Delta: -0.0000 | Null Mean: +0.0000 (std: 0.